# Taller Final — Análisis de Poblaciones desde Reads
## Llamado de variantes y conteo de OTUs

**Curso:** Bioinformática para Ingenieros — Pontificia Universidad Javeriana  
**Docente:** MSc. Juan Sebastián Malagón Torres  
**Duración:** 120 minutos  
**Hardware objetivo:** 2 GB de RAM, 2 núcleos (sin GPU, sin internet obligatorio)

---

### Idea rectora: *"Dos poblaciones, dos granularidades"*

En bioinformática frecuentemente partimos de la misma materia prima: **reads de secuenciación en formato FASTQ**. Sin embargo, según la pregunta biológica, el modelo inferencial cambia por completo:

1. **Caso A — Genómica clínica (*Mycobacterium tuberculosis*):**  
   Sospecha de resistencia a rifampicina, isoniacida y fluoroquinolonas. Se secuenció un **panel dirigido de genes de resistencia** (~93 kb).  
   - **Pregunta:** *¿Qué variantes existen dentro de una población de moléculas?*  
   - **Estrategia:** Alinear contra una **referencia** e inferir **posición por posición** (SNPs e indels → **VCF**).

2. **Caso B — Ecología microbiana (3 comunidades 16S, región V3–V4):**  
   Tres muestras ambientales/clínicas secuenciadas por amplicón 16S.  
   - **Pregunta:** *¿Qué organismos componen cada comunidad y en qué proporción?*  
   - **Estrategia:** Agrupar por **similitud de secuencia (97 %)** sin referencia obligatoria → **Tabla de OTUs** y perfiles de diversidad.

```
FASTQ ──▶ QC (fastp) ──┬──▶ [Alineamiento BWA ──▶ BAM ──▶ bcftools ──▶ VCF]          (Caso A: Variantes)
                       └──▶ [Merge/Filtro ──▶ Clustering 97% ──▶ Quimeras ──▶ OTUs]  (Caso B: Comunidad)
```

> **Pregunta guía:** *¿Cuándo conviene comparar contra un mapa conocido y cuándo conviene agrupar por similitud?*


## 0. Preparación del entorno y verificación de recursos

Antes de lanzar cualquier análisis bioinformático en un equipo con **2 GB de RAM y 2 núcleos**, debemos:
1. Verificar la memoria disponible y los núcleos.
2. Comprobar que todas las herramientas (`fastp`, `bwa`, `samtools`, `bcftools`, `vsearch`) están disponibles.
3. Fijar parámetros globales conservadores (`THREADS = 2`, `SORT_MEM = "64M"`) y ejecutar las herramientas **de forma secuencial**.


In [ ]:
import gc
import gzip
import json
import os
import shutil
import subprocess
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # Backend no interactivo para ahorrar RAM
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Parámetros globales para entorno de 2 GB RAM / 2 cores
THREADS = 2
SORT_MEM = "64M"

# Rutas del taller
BASE_DIR = Path(".").resolve()
DATOS_A = BASE_DIR / "datos" / "caso_A"
DATOS_B = BASE_DIR / "datos" / "caso_B"
RES_DIR = BASE_DIR / "resultados"
RES_A = RES_DIR / "caso_A"
RES_B = RES_DIR / "caso_B"
RES_A.mkdir(parents=True, exist_ok=True)
RES_B.mkdir(parents=True, exist_ok=True)

# Chequeo de hardware
def check_hardware():
    cores = os.cpu_count() or 1
    mem_total_gb, mem_avail_gb = None, None
    meminfo = Path("/proc/meminfo")
    if meminfo.exists():
        info = {}
        for line in meminfo.read_text().splitlines():
            parts = line.split(":")
            if len(parts) == 2:
                val = parts[1].strip().split()[0]
                info[parts[0]] = int(val)  # kB
        mem_total_gb = info.get("MemTotal", 0) / (1024 ** 2)
        mem_avail_gb = info.get("MemAvailable", 0) / (1024 ** 2)
    print(f"Núcleos CPU detectados : {cores}")
    if mem_total_gb is not None:
        print(f"RAM total              : {mem_total_gb:.2f} GB")
        print(f"RAM disponible         : {mem_avail_gb:.2f} GB")
        if mem_avail_gb < 1.2:
            print("⚠️  Advertencia: menos de 1.2 GB disponibles. Cierra otras aplicaciones.")
    print(f"Parámetros fijados     : THREADS={THREADS}, SORT_MEM={SORT_MEM}")

check_hardware()


In [ ]:
# Verificación de herramientas de línea de comandos
herramientas = ["fastp", "bwa", "samtools", "bcftools", "vsearch"]
for tool in herramientas:
    path = shutil.which(tool)
    assert path is not None, f"No se encontró {tool} en el PATH."
    print(f"✅ {tool:<10} -> {path}")

def run_cmd(cmd: str):
    """Ejecuta un comando de shell y muestra un resumen limpio."""
    res = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if res.returncode != 0:
        print("STDERR:", res.stderr[-1000:])
        raise RuntimeError(f"Error ejecutando: {cmd}")
    return res


In [ ]:
# Inventario de datos de entrada
print("=== CASO A (Genómica clínica - M. tuberculosis) ===")
for f in sorted(DATOS_A.iterdir()):
    print(f"  {f.name:<30} {f.stat().st_size / (1024**2):6.2f} MB")

print("\n=== CASO B (Ecología microbiana - 16S V3-V4) ===")
for f in sorted(DATOS_B.iterdir()):
    print(f"  {f.name:<30} {f.stat().st_size / (1024**2):6.2f} MB")


---
## 1. Módulo A — Llamado de variantes (*M. tuberculosis*)

### 1.1 Control de calidad (`fastp`) y alineamiento (`bwa mem` + `samtools`)
Primero limpiamos adaptadores y bases de baja calidad con `fastp`, y luego alineamos los reads pareados contra el panel de referencia `panel_resistencia.fasta`.  
Observa que usamos `samtools sort -@ 2 -m 64M` para evitar que `samtools` reserve 768 MB por hilo por defecto.


In [ ]:
# 1) Control de calidad con fastp
r1_in = DATOS_A / "mtb_panel_R1.fastq.gz"
r2_in = DATOS_A / "mtb_panel_R2.fastq.gz"
r1_qc = RES_A / "mtb_R1.qc.fastq.gz"
r2_qc = RES_A / "mtb_R2.qc.fastq.gz"
fastp_json = RES_A / "fastp_A.json"
fastp_html = RES_A / "fastp_A.html"

run_cmd(
    f"fastp -i {r1_in} -I {r2_in} -o {r1_qc} -O {r2_qc} "
    f"--thread {THREADS} --qualified_quality_phred 20 --length_required 50 "
    f"--json {fastp_json} --html {fastp_html}"
)

qc_data = json.loads(fastp_json.read_text())
before = qc_data["summary"]["before_filtering"]
after = qc_data["summary"]["after_filtering"]
print(f"Reads antes de QC  : {before['total_reads']:,} (Q30: {before['q30_rate']*100:.2f}%)")
print(f"Reads después de QC: {after['total_reads']:,} (Q30: {after['q30_rate']*100:.2f}%)")


In [ ]:
# 2) Indexar referencia y alinear con BWA MEM -> BAM ordenado e indexado
ref_fasta = DATOS_A / "panel_resistencia.fasta"
bam_sorted = RES_A / "mtb_panel.sorted.bam"

if not (DATOS_A / "panel_resistencia.fasta.bwt").exists():
    run_cmd(f"bwa index {ref_fasta}")

run_cmd(
    f"bwa mem -t {THREADS} {ref_fasta} {r1_qc} {r2_qc} | "
    f"samtools sort -@ {THREADS} -m {SORT_MEM} -o {bam_sorted} -"
)
run_cmd(f"samtools index {bam_sorted}")

flagstat = run_cmd(f"samtools flagstat {bam_sorted}").stdout
print(flagstat)


**Interpretación:**  
- Más del **99 %** de los reads mapean al panel y están *properly paired*. Esto confirma que no hay contaminación externa y que el tamaño de inserto es consistente.


### 1.2 ¿Hay suficiente cobertura en todo el panel?
Calculamos la profundidad base por base con `samtools depth -a` y la remuestreamos cada 50 pb para graficar sin saturar la memoria.


In [ ]:
depth_tsv = RES_A / "mtb_depth.tsv"
run_cmd(f"samtools depth -a {bam_sorted} > {depth_tsv}")

depth_df = pd.read_csv(depth_tsv, sep="\t", names=["chrom", "pos", "depth"])
genes_df = pd.read_csv(DATOS_A / "panel_genes.tsv", sep="\t")

print(f"Cobertura media: {depth_df['depth'].mean():.1f}x | Mínima: {depth_df['depth'].min()}x | Máxima: {depth_df['depth'].max()}x")

# Remuestreo cada 50 pb para graficar ligero
depth_sub = depth_df.iloc[::50, :]

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(depth_sub["pos"], depth_sub["depth"], color="#1f77b4", lw=0.9)
ax.axhline(10, color="red", ls="--", lw=1, label="Umbral mínimo DP=10")

for _, row in genes_df.iterrows():
    mid = (row["start"] + row["end"]) / 2
    ax.axvspan(row["start"], row["end"], alpha=0.08, color="gray")
    ax.text(mid, depth_sub["depth"].max() * 0.92, row["gene"], ha="center", fontsize=8, fontweight="bold")

ax.set_title("Cobertura de secuenciación a lo largo del panel de resistencia (M. tuberculosis)")
ax.set_xlabel("Posición en el panel (pb)")
ax.set_ylabel("Profundidad (x)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.savefig(RES_A / "cobertura_panel.png", dpi=130)
plt.show()
plt.close(fig)


**Observación clave:**  
En la zona de *pncA* hay una ventana con cobertura muy baja (~5–15×). En esa región, cualquier variante candidata tendría poca evidencia estadística y alto riesgo de ser un falso positivo o falso negativo.


### 1.3 Del BAM al VCF (`bcftools mpileup` + `bcftools call`)
Ejecutamos `mpileup` en *streaming* (`-Ou`) hacia `bcftools call` para no generar archivos intermedios pesados, y extraemos las columnas clave a TSV con `bcftools query`.


In [ ]:
vcf_raw = RES_A / "mtb_raw.vcf.gz"
vcf_filt = RES_A / "mtb_filtered.vcf.gz"

run_cmd(
    f"bcftools mpileup -f {ref_fasta} -q 20 -Q 20 -a AD,DP --threads {THREADS} -Ou {bam_sorted} | "
    f"bcftools call -mv --threads {THREADS} -Oz -o {vcf_raw}"
)
run_cmd(f"bcftools index {vcf_raw}")

# Filtrado estándar: QUAL >= 30 y DP >= 10
run_cmd(f"bcftools filter -i 'QUAL>=30 && INFO/DP>=10' -Oz -o {vcf_filt} {vcf_raw}")
run_cmd(f"bcftools index {vcf_filt}")

def vcf_to_df(vcf_path: Path) -> pd.DataFrame:
    tsv = run_cmd(
        f"bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%QUAL\t%INFO/DP\t[%AD]\t[%GT]\n' {vcf_path}"
    ).stdout
    rows = []
    for line in tsv.strip().splitlines():
        chrom, pos, ref, alt, qual, dp, ad, gt = line.split("\t")
        ad_parts = [int(x) for x in ad.split(",")]
        af = ad_parts[1] / sum(ad_parts) if sum(ad_parts) > 0 else 0.0
        rows.append({
            "CHROM": chrom, "POS": int(pos), "REF": ref, "ALT": alt,
            "QUAL": float(qual), "DP": int(dp), "AF": round(af, 3), "GT": gt
        })
    return pd.DataFrame(rows)

df_raw = vcf_to_df(vcf_raw)
df_filt = vcf_to_df(vcf_filt)
df_filt_af = df_filt[df_filt["AF"] >= 0.20].copy()

print(f"Variantes crudas               : {len(df_raw)}")
print(f"Variantes QUAL>=30 & DP>=10    : {len(df_filt)}")
print(f"Variantes + AF>=0.20           : {len(df_filt_af)}")
df_filt_af.head(10)


### 1.4 Efecto de los umbrales y comparación con la verdad (`truth.vcf`)
Comparemos qué pasa si relajamos `QUAL >= 20` vs `QUAL >= 30`, y crucemos nuestras variantes filtradas contra `truth.vcf` y las coordenadas de genes (`panel_genes.tsv`).


In [ ]:
# Cargar truth.vcf
truth_rows = []
with open(DATOS_A / "truth.vcf") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.strip().split("\t")
        info_dict = dict(item.split("=") for item in parts[7].split(";") if "=" in item)
        truth_rows.append({
            "POS": int(parts[1]), "REF": parts[3], "ALT": parts[4],
            "TRUE_AF": float(info_dict.get("AF", 0)), "NOTE": info_dict.get("NOTE", "")
        })
df_truth = pd.DataFrame(truth_rows)

# Anotar gen para cada variante detectada
def find_gene(pos):
    hit = genes_df[(genes_df["start"] <= pos) & (genes_df["end"] >= pos)]
    return hit.iloc[0]["gene"] if len(hit) > 0 else "intergenica"

df_filt_af["GENE"] = df_filt_af["POS"].apply(find_gene)

# Comparación con truth.vcf (por posición)
tp = df_filt_af[df_filt_af["POS"].isin(df_truth["POS"])]
fp = df_filt_af[~df_filt_af["POS"].isin(df_truth["POS"])]
fn = df_truth[~df_truth["POS"].isin(df_filt_af["POS"])]

sensibilidad = len(tp) / len(df_truth) * 100
precision = len(tp) / len(df_filt_af) * 100 if len(df_filt_af) > 0 else 0

print(f"Verdaderos Positivos (TP): {len(tp)} / {len(df_truth)}")
print(f"Falsos Positivos (FP)    : {len(fp)}")
print(f"Falsos Negativos (FN)    : {len(fn)}")
print(f"Sensibilidad             : {sensibilidad:.1f}%")
print(f"Precisión                : {precision:.1f}%")

# Mostrar variantes en genes clave de resistencia
res_genes = df_filt_af.merge(df_truth[["POS", "NOTE"]], on="POS", how="left")
print("\n=== Variantes detectadas en rpoB, katG y gyrA ===")
res_genes[res_genes["GENE"].isin(["rpoB", "katG", "gyrA"])][["GENE", "POS", "REF", "ALT", "QUAL", "DP", "AF", "NOTE"]]


**Conclusión biológica (Caso A):**  
- Se detectan claramente las mutaciones responsables de resistencia: ***rpoB* S531L** (rifampicina), ***katG* S315T** (isoniacida) y ***gyrA* D94G** (fluoroquinolonas, en frecuencia alélica intermedia ~0.35–0.45, indicativo de heteroresistencia).  
- Si bajáramos `QUAL` a 20 o elimináramos el filtro `DP >= 10`, entrarían artefactos de secuenciación en la zona de baja cobertura de *pncA*, reduciendo la precisión.


---
## 2. Módulo B — Conteo de OTUs (Comunidades 16S V3–V4)

Ahora cambiamos de granularidad: **no tenemos un genoma de referencia único**, sino amplicones 16S de una comunidad mixta.

### 2.1 Unión de pares (`fastq_mergepairs`) y filtrado de calidad (`fastq_filter`)


In [ ]:
muestras = ["muestra_alta", "muestra_media", "muestra_baja"]
merged_files = []
qc_stats_b = []

for m in muestras:
    r1 = DATOS_B / f"{m}_R1.fastq.gz"
    r2 = DATOS_B / f"{m}_R2.fastq.gz"
    merged = RES_B / f"{m}.merged.fastq"
    filtered = RES_B / f"{m}.filtered.fasta"

    run_cmd(f"vsearch --fastq_mergepairs {r1} --reverse {r2} --fastqout {merged} --threads {THREADS}")
    run_cmd(
        f"vsearch --fastq_filter {merged} --fastq_maxee 1.0 --fastq_minlen 350 "
        f"--fastaout {filtered} --relabel {m}. --sample {m} --threads {THREADS}"
    )

    # Contar secuencias retenidas
    n_filt = sum(1 for line in open(filtered) if line.startswith(">"))
    qc_stats_b.append({"muestra": m, "reads_filtrados": n_filt})
    merged_files.append(filtered)

pd.DataFrame(qc_stats_b)


### 2.2 Dereplicación, agrupamiento al 97 % y eliminación de quimeras
Concatenamos las tres muestras ya etiquetadas (`--relabel`), dereplicamos secuencias idénticas, agrupamos a **97 % de identidad** y removemos quimeras *de novo* con `uchime3_denovo`.


In [ ]:
all_fasta = RES_B / "all_samples.fasta"
derep_fasta = RES_B / "all.derep.fasta"
otus_raw = RES_B / "otus_97_raw.fasta"
otus_nonchim = RES_B / "otus_97.fasta"
otu_table_tsv = RES_B / "otu_table_97.tsv"

# Concatenar FASTA de las 3 muestras
with open(all_fasta, "w") as out_f:
    for f in merged_files:
        out_f.write(f.read_text())

run_cmd(f"vsearch --derep_fulllength {all_fasta} --output {derep_fasta} --sizeout --minuniquesize 2 --threads {THREADS}")
run_cmd(f"vsearch --cluster_size {derep_fasta} --id 0.97 --centroids {otus_raw} --relabel OTU_ --sizein --sizeout --threads {THREADS}")
run_cmd(f"vsearch --uchime3_denovo {otus_raw} --nonchimeras {otus_nonchim} --threads {THREADS}")

n_raw = sum(1 for l in open(otus_raw) if l.startswith(">"))
n_clean = sum(1 for l in open(otus_nonchim) if l.startswith(">"))
print(f"OTUs al 97% antes de quimeras : {n_raw}")
print(f"OTUs al 97% después de quimeras: {n_clean}")

# Construir tabla de OTUs (matriz OTU x muestra)
run_cmd(f"vsearch --usearch_global {all_fasta} --db {otus_nonchim} --id 0.97 --otutabout {otu_table_tsv} --threads {THREADS}")

otu_df = pd.read_csv(otu_table_tsv, sep="\t", index_col=0)
otu_df.index.name = "OTU_ID"
print(f"Dimensiones de la tabla de OTUs: {otu_df.shape}")
otu_df.head(10)


### 2.3 Diversidad alfa y curvas de rarefacción
Calculamos **OTUs observados**, **Shannon ($H'$)**, **Simpson ($1-D$)** y **Chao1** para cada muestra, y graficamos curvas de rarefacción para verificar si el esfuerzo de muestreo fue suficiente.


In [ ]:
def alpha_metrics(counts: np.ndarray) -> dict:
    c = counts[counts > 0]
    total = c.sum()
    p = c / total
    obs = len(c)
    shannon = -np.sum(p * np.log(p))
    simpson = 1.0 - np.sum(p ** 2)
    f1 = np.sum(c == 1)
    f2 = np.sum(c == 2)
    chao1 = obs + (f1 * (f1 - 1)) / (2 * (f2 + 1)) if f1 > 0 else obs
    return {
        "OTUs_observados": obs,
        "Shannon": round(shannon, 3),
        "Simpson": round(simpson, 3),
        "Chao1": round(chao1, 1),
    }

alpha_df = pd.DataFrame({m: alpha_metrics(otu_df[m].values) for m in muestras}).T
display(alpha_df)

# Curvas de rarefacción
rng = np.random.default_rng(42)
depths = np.linspace(500, int(otu_df.sum().min()), 12, dtype=int)

fig, ax = plt.subplots(figsize=(7, 4))
for m in muestras:
    pool = np.repeat(otu_df.index.values, otu_df[m].values)
    richness = []
    for d in depths:
        sub = rng.choice(pool, size=d, replace=False)
        richness.append(len(np.unique(sub)))
    ax.plot(depths, richness, marker="o", label=m)

ax.set_title("Curvas de rarefacción por muestra")
ax.set_xlabel("Número de reads muestreados")
ax.set_ylabel("OTUs observados")
ax.legend()
plt.tight_layout()
plt.savefig(RES_B / "rarefaccion.png", dpi=130)
plt.show()
plt.close(fig)


### 2.4 Composición relativa y diversidad beta (Bray-Curtis + PCoA)


In [ ]:
# Abundancia relativa
otu_rel = otu_df.div(otu_df.sum(axis=0), axis=1)

# Barplot de composición (Top 10 OTUs + Otros)
top10 = otu_rel.mean(axis=1).sort_values(ascending=False).head(10).index
plot_df = otu_rel.loc[top10].copy()
plot_df.loc["Otros"] = 1.0 - plot_df.sum(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
plot_df.T.plot(kind="bar", stacked=True, ax=axes[0], colormap="tab20")
axes[0].set_title("Composición relativa (Top 10 OTUs)")
axes[0].set_ylabel("Abundancia relativa")
axes[0].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)

# Distancia de Bray-Curtis y PCoA con numpy.linalg.eigh
n_m = len(muestras)
bc = np.zeros((n_m, n_m))
for i in range(n_m):
    for j in range(n_m):
        u, v = otu_rel[muestras[i]].values, otu_rel[muestras[j]].values
        bc[i, j] = np.sum(np.abs(u - v)) / np.sum(u + v)

# Doble centrado para PCoA
D2 = bc ** 2
J = np.eye(n_m) - np.ones((n_m, n_m)) / n_m
B = -0.5 * J @ D2 @ J
eigvals, eigvecs = np.linalg.eigh(B)
idx = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]
coords = eigvecs[:, :2] * np.sqrt(np.maximum(eigvals[:2], 0))

for i, m in enumerate(muestras):
    axes[1].scatter(coords[i, 0], coords[i, 1], s=120, label=m)
    axes[1].text(coords[i, 0] + 0.01, coords[i, 1] + 0.01, m, fontsize=9)
axes[1].set_title("PCoA (Distancia de Bray-Curtis)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

plt.tight_layout()
plt.savefig(RES_B / "composicion_y_pcoa.png", dpi=130)
plt.show()
plt.close(fig)


### 2.5 Efecto del umbral de identidad (95 % vs 97 % vs 99 %)
¿Qué ocurre si cambiamos el umbral de agrupamiento de 97 % a 95 % (nivel de género aproximado) o 99 % (sub-especie / casi ASV)?


In [ ]:
umbrales = [0.95, 0.97, 0.99]
otus_por_umbral = []

for pid in umbrales:
    out_c = RES_B / f"otus_{int(pid*100)}.fasta"
    run_cmd(f"vsearch --cluster_size {derep_fasta} --id {pid} --centroids {out_c} --sizein --sizeout --threads {THREADS}")
    n_c = sum(1 for l in open(out_c) if l.startswith(">"))
    otus_por_umbral.append({"identidad_%": int(pid * 100), "OTUs": n_c})

df_umbrales = pd.DataFrame(otus_por_umbral)
display(df_umbrales)

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(df_umbrales["identidad_%"], df_umbrales["OTUs"], marker="o", lw=2, color="#2ca02c")
ax.set_title("Número de OTUs según umbral de identidad")
ax.set_xlabel("Identidad de secuencia (%)")
ax.set_ylabel("OTUs generados")
ax.set_xticks([95, 97, 99])
plt.tight_layout()
plt.savefig(RES_B / "umbrales_otu.png", dpi=130)
plt.show()
plt.close(fig)

gc.collect()


---
## 3. Cierre integrador: Variante vs. OTU

| Dimensión | Llamado de Variantes (Caso A) | Conteo de OTUs (Caso B) |
|---|---|---|
| **Pregunta biológica** | ¿Qué mutaciones existen respecto a una referencia? | ¿Qué organismos componen la comunidad y en qué proporción? |
| **Unidad mínima** | Posición nucleotídica (SNP / indel) | Clúster de secuencias (OTU al 97 %) |
| **¿Requiere referencia?** | Sí (genoma o panel) | No obligatoria (agrupamiento *de novo*) |
| **Archivo de salida** | VCF (`.vcf.gz`) | Tabla de OTUs (`.tsv`) |
| **Umbral crítico** | `QUAL`, `DP`, `AF` | Identidad de secuencia (95 / 97 / 99 %) |
| **Cómo se valida** | Cobertura, Ts/Tv, `truth.vcf` / Sanger | Quimeras, rarefacción, comunidades *mock* |

### Preguntas de discusión en clase
1. **Concepto:** *¿Por qué no usamos `bwa mem` + `bcftools` para el Caso B de 16S?*  
   Porque en una comunidad ambiental no conocemos de antemano todas las especies presentes; alinear contra un único genoma descartaría la mayoría de los reads o los forzaría a mapear mal.
2. **Interpretación:** *En el Caso A, ¿por qué `gyrA` D94G tiene AF ≈ 0.40–0.45 mientras `rpoB` S531L tiene AF ≈ 0.90?*  
   Indica **heteroresistencia** (subpoblación resistente coexistiendo con población sensible), un fenómeno clínico clave en tuberculosis.
3. **Decisión metodológica:** *Si pasamos de 97 % a 99 % en OTUs, ¿por qué sube el número de OTUs?*  
   Al exigir mayor similitud, errores de secuenciación o microvariación intra-especie separan una misma especie en múltiples clústeres (**sobreseparación**).
